# 第 1 章实践：检验你的基础认知

> 本章考核 = 客观题；三道编程题为**选做体验**（不计入考核），动手完成对 01.06 的理解会更深。

本节以**客观题**作为 L4-50 的正式考核（选择题/填空）。三道编程实践**不纳入考核**，定位为选做体验题——想动手的学习者基于 01.06 的向量加法闭环模板完成即可，全部附参考答案。建议流程：先做客观题检验概念，再用实践题检验手感。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 客观题

1. （选择）神经网络计算负载的典型特征是：
   A. 分支密集、串行依赖强<br/>
   B. 海量数据上的少数重复操作<br/>
   C. 以标量计算为主
2. （选择）矩阵乘的操作数存放在哪些存储中，由什么单元读取？
   A. UB，Vector 单元<br/>
   B. L0A/L0B Buffer，Cube 单元<br/>
   C. GM 直接读取，Scalar 单元
3. （填空）PyPTO Pro 与 Ascend C 的本质区别：Pro 含 ______ 语义描述能力、具有 ______ 属性，Ascend C 归根结底是 ______。
4. （选择）"对一整块数据执行同一种操作"的直觉对应哪种范式？
   A. SIMT<br/>B. SIMD<br/>C. SPMD
5. （判断）`TORCH_DEVICE_BACKEND_AUTOLOAD` 与 `TILE_FWK_DEVICE_ID` 是本课程运行代码前需要设置的环境变量。（对/错）

In [ ]:
!cat ./answer/01.07_objective_answer.txt

## 2. 选做体验一（简单）：修改规格重跑向量加法

**任务**：把 01.06 的向量加法算子改为处理 Shape `[16, 512]`、dtype `FP16` 的输入。

**要求**：

1. 修改 kernel 参数类型标注与 TileType 定义；
2. 重新规划 `make_tile_group` 的片上地址（提示：先计算单个 Tile 的字节数）；
3. 与 `torch.add` 的结果做精度校验（FP16 建议使用 `rtol=1e-3, atol=1e-3`）。

请在下方代码单元完成你的实现：

In [ ]:
# 在此完成实践一：Shape [16, 512] / FP16 的向量加法
# 提示：参考 01.06 的完整闭环模板
# import pypto_pro.language as pl
# ...
pass

## 3. 选做体验二（中等）：标量缩放算子

**任务**：实现 `out = alpha * x`，其中 alpha 是运行时标量（FP32），x 是 Shape `[8, 256]` 的 FP32 Tensor。

**提示**：

1. Kernel 参数中用 `pl.DT_FP32` 类型标注声明运行时标量；
2. Tile 级缩放可使用 `pl.mul` 接口（接口语义：逐元素乘），传入 Tile 与标量参数的用法请查阅 API 文档确认——**查阅文档而不是猜**，是本课程强调的习惯；
3. alpha 的数值由 Host 侧在调用 Kernel 时传入。

请在下方代码单元完成：

In [ ]:
# 在此完成实践二：out = alpha * x
pass

## 4. 选做体验三（困难）：双输入线性组合

**任务**：实现 `out = alpha * x + beta * y`，alpha/beta 均为运行时标量，x/y 为 `[8, 256]` FP32。

**提示**：

1. 两个运行时标量 + 三个 Tensor（两入一出）共 5 个 Kernel 参数，注意类型标注顺序与调用顺序一致；
2. Tile 侧思考：可以用一个中间 Tile 保存 `alpha*x` 的结果，再累加 `beta*y`（接口组合不止一种，说明你的选择理由）；
3. 用 PyTorch 表达式 `alpha * x + beta * y` 生成参考结果校验。

请在下方代码单元完成：

In [ ]:
# 在此完成实践三：out = alpha * x + beta * y
pass

完成后查看参考答案，对照三个要点：类型标注是否完整、地址规划是否按字节数推算、精度阈值选择是否合理。

In [ ]:
!cat ./answer/01.07_practice_answer.txt

### 完整参考实现与验收

三个体验任务分别对应 `add_fp16`、`scale`、`linear_combination`。全部使用单核；标量用例覆盖零、负数与非整数，避免只测 alpha=1 而漏掉传参错误。

[参考源码](./src/first_kernels.py) 可在 950 上从章节目录执行 `python src/first_kernels.py`。下列单元仅展示源码；设备编译、精度与性能均以实际执行为准。


In [ ]:
!cat ./src/first_kernels.py


## 5. 本节小结

三道实践覆盖了本章全部动手要点：**规格定义 → 参数类型标注 → Tile/地址规划 → 接口选择 → Host 校验**。如果你顺利完成，说明已具备进入第 2 章（SIMD 编程模型）的条件；如果某道实践卡住，建议回到对应小节（01.03 存储层级 / 01.06 完整闭环）复习后再继续。

下一章我们将深入 SIMD 编程模型：核函数、数据对象、内存层级、搬运、NBuffer 与同步机制——把"跑通"升级为"理解每一行为什么这样写"。